In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az 

# define random seed for reproducibility
random_seed = 42

In [2]:
# read file
df = pd.read_csv('data/scr_brain_group.csv')
# only for no shock
# %% amygdala-hippocampus coupling pymc model
# Encode 'sub' as integer indices
df['sub_idx'] = pd.Categorical(df['sub']).codes
n_subs = df['sub_idx'].nunique()

# Encode 'group' as integer indices (make ordering explicit!)
# Data uses: HC (healthy controls), VCC (combat controls), VPTSD (PTSD)
group_order = ['HC', 'VCC', 'VPTSD']
df['group'] = pd.Categorical(df['group'], categories=group_order, ordered=True)
df['group_idx'] = df['group'].cat.codes
n_groups = df['group_idx'].nunique()

# Check which group is reference (index 0)
print("Group coding (0 = reference):", {g: i for i, g in enumerate(df['group'].cat.categories)})

# Extract variables
pe = df['pe'].values
coupling = df['coupling'].values
amg = df['amg'].values
trialNo = df['trialNo'].values
sub_idx = df['sub_idx'].values
group_idx = df['group_idx'].values


Group coding (0 = reference): {'HC': 0, 'VCC': 1, 'VPTSD': 2}


In [ ]:
with pm.Model() as model_pe_group:
    #beta_amg     = pm.Normal('beta_amg', 0, 1)
    beta_trialNo = pm.Normal('beta_trialNo', 0, 1)
    beta_group_raw = pm.Normal('beta_group_raw', 0, 1, shape=n_groups-1)
    beta_group = pm.math.concatenate([[0], beta_group_raw])
    # Group × Trial: does the trial trajectory differ by group?
    beta_gxt_raw = pm.Normal('beta_gxt_raw', 0, 1, shape=n_groups-1)
    beta_gxt = pm.math.concatenate([[0], beta_gxt_raw])

    mu_a = pm.Normal('mu_a', 0, 1); sigma_a = pm.HalfNormal('sigma_a', 1)
    z_a = pm.Normal('z_a', 0, 1, shape=n_subs)
    a = pm.Deterministic('a', mu_a + z_a*sigma_a)

    mu = a[sub_idx] + beta_group[group_idx] +  beta_trialNo*trialNo + beta_gxt[group_idx]*trialNo

    sigma = pm.HalfNormal('sigma', 1)
    pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
    trace_pe_traj = pm.sample(chains=4, random_seed=random_seed,
                               return_inferencedata=True,
                               idata_kwargs={"log_likelihood": True})
az.summary(trace_pe_traj, var_names=['beta_group_raw', 'beta_trialNo', 'beta_gxt_raw'], hdi_prob=0.89)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_trialNo, beta_group_raw, beta_gxt_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 8 seconds.


,mean,sd,hdi_5.5%,hdi_94.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta_group_raw[0],0.001,0.033,-0.052,0.053,0.001,0.0,1307.0,2231.0,1.01
beta_group_raw[1],-0.002,0.034,-0.053,0.054,0.001,0.0,1255.0,1894.0,1.01
beta_trialNo,0.000,0.001,-0.001,0.001,0.000,0.0,1052.0,2029.0,1.01
beta_gxt_raw[0],-0.000,0.001,-0.001,0.001,0.000,0.0,1290.0,2085.0,1.01
beta_gxt_raw[1],-0.000,0.001,-0.001,0.001,0.000,0.0,1149.0,2167.0,1.01


In [4]:
group_names = ["HC", "VCC", "VPTSD"]
def posterior_contrast_summary(idata, group_names, hdi_prob=0.89):
    posterior = idata.posterior
    # beta_group_raw has coefficients for groups 1..n,
    # while group 0 is the reference and therefore exactly 0
    beta_raw = posterior["beta_group_raw"].values
    # shape: chains x draws x (n_groups-1)
    chains, draws, _ = beta_raw.shape
    # reconstruct full beta_group posterior
    reference = np.zeros((chains, draws, 1))
    beta_group = np.concatenate([reference, beta_raw], axis=2)
    results = []
    for i in range(len(group_names)):
        for j in range(i + 1, len(group_names)):
            # i - j
            contrast = beta_group[:, :, i] - beta_group[:, :, j]
            samples = contrast.ravel()
            hdi = az.hdi(samples, hdi_prob=hdi_prob)
            # Probability of direction
            pdirection = max(
                np.mean(samples > 0),
                np.mean(samples < 0)
            )
            results.append({
                "contrast": f"{group_names[i]} - {group_names[j]}",
                "beta": np.mean(samples),
                "SD": np.std(samples),
                "HDI_5.5%": hdi[0],
                "HDI_94.5%": hdi[1],
                "pd": pdirection
            })
    return pd.DataFrame(results)
group_contrasts = posterior_contrast_summary(
    trace_pe_traj,
    group_names,
    hdi_prob=0.89
)
print(group_contrasts.round(4))

      contrast    beta      SD  HDI_5.5%  HDI_94.5%      pd
0     HC - VCC -0.0011  0.0329   -0.0530     0.0521  0.5112
1   HC - VPTSD  0.0018  0.0339   -0.0538     0.0533  0.5215
2  VCC - VPTSD  0.0029  0.0320   -0.0482     0.0542  0.5352
